# 06d — Reentrenamiento con lote activo

## Objetivo
Combinar las 180 etiquetas canónicas con las 300 anotaciones del lote activo y comparar el baseline de palabras con un modelo interpretable de TF-IDF de palabras y caracteres.

## Diseño de evaluación
- Validación cruzada estratificada y agrupada por `source_post_id`.
- Cinco folds y cinco repeticiones.
- Dos perfiles de decisión: `balanced` (máximo F1 positivo) y `high_precision` (precisión mínima exploratoria de 80%).
- Ninguna predicción sobre el corpus completo se interpreta como prevalencia.

## Entradas
- `data/processed/manual_review_training_combined_v2.csv`
- `data/processed/x_media_anchored_interactions_corpus_formal_with_hostility_and_experimental_hate_predictions.csv`
- `reports/formal_ml/hate_false_negatives_for_review.csv`

## Salidas
- Métricas y matrices en `reports/formal_ml/`.
- Modelos versionados en `models/formal/`.
- Corpus con predicciones v2 en `data/processed/`.


In [ ]:
import importlib
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start):
    for candidate in [start] + list(start.parents):
        if (candidate / "config").exists() and (candidate / "src").exists():
            return candidate
        child = candidate / "HateCR"
        if (child / "config").exists() and (child / "src").exists():
            return child
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.labels as label_utils
import src.modeling as modeling
import src.model_refinement as refinement
import src.model_history as model_history
import src.model_visualization as model_viz
importlib.reload(label_utils)
importlib.reload(modeling)
importlib.reload(refinement)
importlib.reload(model_history)
importlib.reload(model_viz)

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports" / "formal_ml"
FIGURES_DIR = REPORTS_DIR / "figures"
MODELS_DIR = PROJECT_ROOT / "models" / "formal"
for directory in [REPORTS_DIR, FIGURES_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

TRAINING_PATH = DATA_PROCESSED / "manual_review_training_combined_v2.csv"
CORPUS_PATH = DATA_PROCESSED / "x_media_anchored_interactions_corpus_formal_with_hostility_and_experimental_hate_predictions.csv"
KNOWN_FN_PATH = REPORTS_DIR / "hate_false_negatives_for_review.csv"
OUTPUT_CORPUS_PATH = DATA_PROCESSED / "x_media_anchored_interactions_corpus_formal_predictions_v2.csv"
INITIAL_MANUAL_SAMPLE_PATH = PROJECT_ROOT / "reports" / "formal_eda" / "manual_review_sample.csv"
INITIAL_HELDOUT_METRICS_PATH = REPORTS_DIR / "baseline_ml_metrics.csv"

RANDOM_STATE = int(os.getenv("RANDOM_STATE", "42"))
N_SPLITS = int(os.getenv("REFINEMENT_CV_SPLITS", "5"))
N_REPEATS = int(os.getenv("REFINEMENT_CV_REPEATS", "5"))
MIN_PRECISION = float(os.getenv("REFINEMENT_MIN_PRECISION", "0.80"))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CV:", N_SPLITS, "folds x", N_REPEATS, "repeticiones")
print("Precisión mínima perfil high_precision:", MIN_PRECISION)
print("API y descargas: desactivadas")


In [ ]:
def read_csv_ids(path, name, id_columns):
    if not path.exists():
        raise FileNotFoundError(f"Falta {name}: {path}")
    frame = pd.read_csv(
        path,
        dtype={column: "string" for column in id_columns},
        low_memory=False,
    )
    print(f"[OK] {name}: {len(frame):,} filas")
    return frame


training_df = read_csv_ids(
    TRAINING_PATH,
    "entrenamiento combinado v2",
    ["review_id", "tweet_id", "source_post_id"],
)
corpus_df = read_csv_ids(
    CORPUS_PATH,
    "corpus formal",
    ["tweet_id", "anchor_post_id"],
)
known_fn_df = read_csv_ids(
    KNOWN_FN_PATH,
    "falsos negativos v1",
    ["review_id", "tweet_id", "source_post_id"],
)

training_df, label_diagnostics = label_utils.prepare_manual_annotations(
    training_df,
    strict=True,
)
if not training_df["annotation_ready_for_training"].all():
    raise ValueError("El conjunto combinado contiene etiquetas incompletas")
if training_df["tweet_id"].nunique() != len(training_df):
    raise ValueError("El conjunto combinado contiene tweet_id duplicados")

training_df["text_model"] = training_df["text"].map(modeling.normalize_text_for_model)
corpus_df["text_model_v2"] = corpus_df["text"].map(modeling.normalize_text_for_model)

label_summary = pd.DataFrame([
    {"metric": "training_rows", "value": len(training_df)},
    {"metric": "source_post_groups", "value": training_df["source_post_id"].nunique()},
    {"metric": "hostility_positive", "value": int(training_df["y_hostility"].sum())},
    {"metric": "hostility_negative", "value": int((training_df["y_hostility"] == 0).sum())},
    {"metric": "hate_positive", "value": int(training_df["y_hate_speech"].sum())},
    {"metric": "hate_negative", "value": int((training_df["y_hate_speech"] == 0).sum())},
    {"metric": "annotation_warnings", "value": int(training_df["annotation_warnings"].ne("").sum())},
])
label_summary.to_csv(REPORTS_DIR / "model_refinement_v2_label_summary.csv", index=False)
display(label_summary)


## Evaluación agrupada
Los comentarios pertenecientes al mismo post madre permanecen juntos dentro de cada fold. Esto reduce la posibilidad de que textos de una misma conversación aparezcan simultáneamente en entrenamiento y evaluación.

Se comparan los mismos pliegues agrupados para cuatro estrategias: lexicon_hits, logreg_tfidf, linearsvc_tfidf y word_char_logreg. El baseline de lexicón predice positivo cuando categorized_lexicon_hit_count > 0; no constituye por sí mismo una clasificación de hostilidad u odio.


In [ ]:
(
    fold_metrics_df,
    cv_summary_df,
    consensus_df,
    threshold_table_df,
    selected_thresholds_df,
    operating_metrics_df,
) = refinement.evaluate_combined_training_set(
    training_df,
    text_column="text_model",
    group_column="source_post_id",
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE,
    min_precision=MIN_PRECISION,
)

fold_metrics_df.to_csv(REPORTS_DIR / "model_refinement_v2_cv_folds.csv", index=False)
cv_summary_df.to_csv(REPORTS_DIR / "model_refinement_v2_cv_summary.csv", index=False)
consensus_df.to_csv(REPORTS_DIR / "model_refinement_v2_cv_consensus.csv", index=False)
threshold_table_df.to_csv(REPORTS_DIR / "model_refinement_v2_threshold_grid.csv", index=False)
selected_thresholds_df.to_csv(REPORTS_DIR / "model_refinement_v2_selected_thresholds.csv", index=False)
operating_metrics_df.to_csv(REPORTS_DIR / "model_refinement_v2_operating_metrics.csv", index=False)

baseline_metrics_df, baseline_confusion_df = (
    refinement.summarize_default_model_consensus(consensus_df)
)
baseline_metrics_df.to_csv(
    REPORTS_DIR / "model_refinement_v2_baseline_metrics.csv", index=False
)
baseline_confusion_df.to_csv(
    REPORTS_DIR / "model_refinement_v2_baseline_confusion_counts.csv", index=False
)

comparison_columns = [
    "target", "model", "precision_positive_mean", "recall_positive_mean",
    "f1_positive_mean", "macro_f1_mean", "balanced_accuracy_mean",
    "average_precision_mean",
]
display(cv_summary_df[comparison_columns])
print("Métricas de consenso fuera de muestra con umbral por defecto:")
display(baseline_metrics_df)
display(operating_metrics_df)


In [ ]:
consensus_profiles_df = refinement.add_profile_predictions(
    consensus_df,
    selected_thresholds_df,
)
consensus_profiles_df.to_csv(
    REPORTS_DIR / "model_refinement_v2_consensus_profiles.csv",
    index=False,
)

confusion_rows = []
for target in ["hostility", "hate"]:
    for profile in ["balanced", "high_precision"]:
        output_path = FIGURES_DIR / f"confusion_{target}_v2_{profile}.png"
        matrix = refinement.save_confusion_figure(
            consensus_profiles_df,
            target=target,
            profile=profile,
            output_path=output_path,
        )
        for actual in [0, 1]:
            for predicted in [0, 1]:
                confusion_rows.append({
                    "target": target,
                    "profile": profile,
                    "actual": actual,
                    "predicted": predicted,
                    "n_rows": int(matrix[actual, predicted]),
                })
confusion_df = pd.DataFrame(confusion_rows)
confusion_df.to_csv(REPORTS_DIR / "model_refinement_v2_confusion_counts.csv", index=False)

plot_df = cv_summary_df.copy()
model_labels = {
    "lexicon_hits": "Lexicón",
    "logreg_tfidf": "Logistic Regression",
    "linearsvc_tfidf": "LinearSVC",
    "word_char_logreg": "Logistic Regression word+char",
}
target_labels = {"hostility": "Hostilidad", "hate": "Odio"}
plot_df["label"] = (
    plot_df["target"].map(target_labels)
    + " | "
    + plot_df["model"].map(model_labels)
)
plot_df = plot_df.sort_values("f1_positive_mean")
colors = plot_df["model"].map({
    "lexicon_hits": "#9A8F76",
    "logreg_tfidf": "#4F758B",
    "linearsvc_tfidf": "#D08C60",
    "word_char_logreg": "#B24632",
})
fig, ax = plt.subplots(figsize=(11, 7.2))
ax.barh(plot_df["label"], plot_df["f1_positive_mean"], color=colors)
ax.errorbar(
    plot_df["f1_positive_mean"],
    plot_df["label"],
    xerr=plot_df["f1_positive_std"].fillna(0),
    fmt="none",
    ecolor="#333333",
    capsize=4,
)
ax.set_xlim(0, 1)
ax.set_xlabel("F1 positivo medio ± desviación estándar")
ax.set_title("Baselines v2: lexicón, Logistic Regression y LinearSVC")
fig.tight_layout()
comparison_figure_path = FIGURES_DIR / "model_refinement_v2_f1_comparison.png"
fig.savefig(comparison_figure_path, dpi=180, bbox_inches="tight")
plt.show()
print("[OK]", comparison_figure_path)

metric_plot_df = baseline_metrics_df.melt(
    id_vars=["target", "model"],
    value_vars=["precision_positive", "recall_positive", "f1_positive"],
    var_name="metric",
    value_name="value",
)
metric_labels = {
    "precision_positive": "Precisión",
    "recall_positive": "Recall",
    "f1_positive": "F1",
}
fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)
for ax, target in zip(axes, ["hostility", "hate"]):
    target_df = metric_plot_df[metric_plot_df["target"].eq(target)]
    models = list(model_labels)
    x = np.arange(len(models))
    width = 0.24
    for offset, metric in enumerate(metric_labels):
        values = (
            target_df[target_df["metric"].eq(metric)]
            .set_index("model")
            .reindex(models)["value"]
            .fillna(0)
            .to_numpy()
        )
        ax.bar(
            x + (offset - 1) * width, values, width,
            label=metric_labels[metric],
        )
    ax.set_xticks(
        x, [model_labels[model] for model in models], rotation=22, ha="right"
    )
    ax.set_ylim(0, 1)
    ax.set_title(target_labels[target])
    ax.set_ylabel("Métrica fuera de muestra")
    ax.grid(axis="y", alpha=0.2)
axes[1].legend(frameon=False, loc="upper right")
fig.suptitle("Comparación de precisión, recall y F1 por baseline v2", y=1.02)
fig.tight_layout()
baseline_metrics_figure_path = (
    FIGURES_DIR / "model_refinement_v2_baseline_metrics_comparison.png"
)
fig.savefig(baseline_metrics_figure_path, dpi=180, bbox_inches="tight")
plt.show()
print("[OK]", baseline_metrics_figure_path)


## Referencia histórica del primer baseline

Este bloque reconstruye las métricas iniciales sin mezclarlas con el entrenamiento v2. LinearSVC y Logistic Regression proceden del holdout original de 45 casos; el lexicón se evalúa sobre las 180 anotaciones.

La variabilidad se estima con 5 pliegues y 5 repeticiones siguiendo el diseño estratificado original. No agrupa por post madre, por lo que es una auditoría histórica y no sustituye la validación agrupada del modelo v2.


In [ ]:
initial_manual_df = read_csv_ids(
    INITIAL_MANUAL_SAMPLE_PATH,
    "muestra manual inicial",
    ["review_id", "tweet_id", "source_post_id"],
)
initial_manual_df, initial_label_diagnostics = (
    label_utils.prepare_manual_annotations(initial_manual_df, strict=True)
)
if not initial_manual_df["annotation_ready_for_training"].all():
    raise ValueError("La muestra inicial contiene etiquetas incompletas")

initial_heldout_metrics_df = pd.read_csv(INITIAL_HELDOUT_METRICS_PATH)
initial_reference_metrics_df = model_history.build_initial_reference_metrics(
    initial_manual_df,
    initial_heldout_metrics_df,
)
initial_reference_metrics_df.to_csv(
    REPORTS_DIR / "initial_baseline_metrics_reference.csv",
    index=False,
)

(
    initial_variability_folds_df,
    initial_variability_summary_df,
) = model_history.run_initial_repeated_cv(
    initial_manual_df,
    n_splits=5,
    n_repeats=5,
    random_state=RANDOM_STATE,
)
initial_variability_display_df = model_history.build_variability_display_table(
    initial_variability_summary_df
)
initial_variability_folds_df.to_csv(
    REPORTS_DIR / "initial_baseline_variability_folds.csv",
    index=False,
)
initial_variability_summary_df.to_csv(
    REPORTS_DIR / "initial_baseline_variability_summary.csv",
    index=False,
)
initial_variability_display_df.to_csv(
    REPORTS_DIR / "initial_baseline_variability_display.csv",
    index=False,
)

model_viz.save_initial_reference_metrics_figure(
    initial_reference_metrics_df,
    FIGURES_DIR / "initial_baseline_metrics_comparison.png",
)
model_viz.save_initial_variability_figure(
    initial_variability_summary_df,
    FIGURES_DIR / "initial_baseline_f1_variability.png",
)

reference_columns = [
    "target_label", "model_label", "evaluation_design", "n_eval",
    "accuracy", "precision_positive", "recall_positive",
    "f1_positive", "macro_f1",
]
display(initial_reference_metrics_df[reference_columns])
display(initial_variability_display_df)
print("[OK] cuadro histórico y análisis de variabilidad")


## Entrenamiento final y aplicación
Los modelos finales se entrenan con las 480 etiquetas, pero conservan nombres y archivos versionados. Los perfiles balanceado y de alta precisión usan el mismo score y diferentes umbrales.


In [ ]:
threshold_map = refinement.selected_threshold_map(selected_thresholds_df)
final_models = {}
model_paths = {}
for target, target_column in refinement.TARGET_COLUMNS.items():
    model = modeling.build_word_char_logreg_pipeline(random_state=RANDOM_STATE)
    model.fit(
        training_df["text_model"].values,
        training_df[target_column].astype(int).values,
    )
    model_path = MODELS_DIR / f"{target}_word_char_logreg_v2.joblib"
    joblib.dump(model, model_path)
    final_models[target] = model
    model_paths[target] = str(model_path)
    print("[OK]", model_path)

metadata = {
    "status": "exploratory_v2",
    "warning": (
        "Training combines a lexicon-enriched canonical sample and an active-learning "
        "batch. Metrics and corpus predictions do not estimate prevalence."
    ),
    "training_path": str(TRAINING_PATH),
    "n_training_rows": int(len(training_df)),
    "n_source_post_groups": int(training_df["source_post_id"].nunique()),
    "class_distribution": {
        "hostility": {
            str(int(key)): int(value)
            for key, value in training_df["y_hostility"].value_counts().sort_index().items()
        },
        "hate": {
            str(int(key)): int(value)
            for key, value in training_df["y_hate_speech"].value_counts().sort_index().items()
        },
    },
    "features": "word_tfidf_1_2_plus_char_wb_tfidf_3_5",
    "classifier": "logistic_regression_class_weight_balanced",
    "cross_validation": {
        "type": "Repeated StratifiedGroupKFold",
        "group": "source_post_id",
        "n_splits": N_SPLITS,
        "n_repeats": N_REPEATS,
        "random_state": RANDOM_STATE,
    },
    "thresholds": threshold_map,
    "selected_threshold_details": json.loads(selected_thresholds_df.to_json(orient="records")),
    "model_paths": model_paths,
    "trained_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
}
metadata_path = MODELS_DIR / "model_refinement_v2_metadata.json"
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print("[OK]", metadata_path)


In [ ]:
corpus_v2_df = corpus_df.copy()
for target, model in final_models.items():
    scores = model.predict_proba(corpus_v2_df["text_model_v2"].values)[:, 1]
    if target == "hostility":
        score_column = "ml_hostility_score_v2"
        balanced_column = "ml_hostility_pred_v2_balanced"
        high_precision_column = "ml_hostility_pred_v2_high_precision"
    else:
        score_column = "ml_hate_speech_score_v2_experimental"
        balanced_column = "ml_hate_speech_pred_v2_balanced_experimental"
        high_precision_column = "ml_hate_speech_pred_v2_high_precision_experimental"
    corpus_v2_df[score_column] = scores
    corpus_v2_df[balanced_column] = (
        scores >= threshold_map[target]["balanced"]
    ).astype(int)
    corpus_v2_df[high_precision_column] = (
        scores >= threshold_map[target]["high_precision"]
    ).astype(int)

corpus_v2_df["ml_model_v2"] = "word_char_logreg_v2"
corpus_v2_df["ml_training_rows_v2"] = len(training_df)
corpus_v2_df.to_csv(OUTPUT_CORPUS_PATH, index=False)

prediction_columns = [
    "ml_hostility_pred_v2_balanced",
    "ml_hostility_pred_v2_high_precision",
    "ml_hate_speech_pred_v2_balanced_experimental",
    "ml_hate_speech_pred_v2_high_precision_experimental",
]
distribution_rows = []
for column in prediction_columns:
    positives = int(corpus_v2_df[column].sum())
    distribution_rows.append({
        "prediction_column": column,
        "n_rows": len(corpus_v2_df),
        "predicted_positive": positives,
        "predicted_positive_pct": round(100 * positives / len(corpus_v2_df), 2),
    })
prediction_distribution_df = pd.DataFrame(distribution_rows)
prediction_distribution_df.to_csv(
    REPORTS_DIR / "model_refinement_v2_corpus_prediction_distribution.csv",
    index=False,
)
print("[OK]", OUTPUT_CORPUS_PATH)
display(prediction_distribution_df)


## Diagnóstico gráfico del porcentaje predicho de odio

El perfil balanceado prioriza el equilibrio entre precisión y recall. El perfil de alta precisión es más conservador. Los porcentajes describen predicciones del modelo sobre este corpus y no prevalencia confirmada de discurso de odio.


In [ ]:
HATE_PROFILE_COLUMNS = {
    "balanced": "ml_hate_speech_pred_v2_balanced_experimental",
    "high_precision": "ml_hate_speech_pred_v2_high_precision_experimental",
}
HATE_MEDIA_MIN_ROWS = int(os.getenv("HATE_MEDIA_MIN_ROWS", "50"))

hate_overall_df = model_viz.summarize_prediction_profiles(
    corpus_v2_df, HATE_PROFILE_COLUMNS
)
hate_overall_df.to_csv(
    REPORTS_DIR / "model_refinement_v2_hate_overall.csv", index=False
)

event_column = "event_name" if "event_name" in corpus_v2_df.columns else "event_id"
hate_by_event_df = model_viz.summarize_profiles_by_group(
    corpus_v2_df, event_column, HATE_PROFILE_COLUMNS
)
hate_by_event_df.to_csv(
    REPORTS_DIR / "model_refinement_v2_hate_by_event.csv", index=False
)

hate_by_media_df = model_viz.summarize_profiles_by_group(
    corpus_v2_df, "anchor_media_handle", HATE_PROFILE_COLUMNS
)
hate_by_media_df.to_csv(
    REPORTS_DIR / "model_refinement_v2_hate_by_media.csv", index=False
)

hate_overlap_df = model_viz.summarize_hostility_hate_overlap(
    corpus_v2_df,
    hostility_column="ml_hostility_pred_v2_balanced",
    hate_column="ml_hate_speech_pred_v2_balanced_experimental",
)
hate_overlap_df.to_csv(
    REPORTS_DIR / "model_refinement_v2_hostility_hate_overlap.csv", index=False
)

quantiles = [0, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1]
hate_score_quantiles_df = pd.DataFrame({
    "quantile": quantiles,
    "hate_score_v2": corpus_v2_df[
        "ml_hate_speech_score_v2_experimental"
    ].quantile(quantiles).to_numpy(),
})
hate_score_quantiles_df.to_csv(
    REPORTS_DIR / "model_refinement_v2_hate_score_quantiles.csv", index=False
)

model_viz.save_overall_profile_figure(
    hate_overall_df,
    FIGURES_DIR / "hate_v2_overall_percentage.png",
)
model_viz.save_score_distribution_figure(
    corpus_v2_df["ml_hate_speech_score_v2_experimental"],
    threshold_map["hate"],
    FIGURES_DIR / "hate_v2_score_distribution.png",
)
model_viz.save_group_profile_figure(
    hate_by_event_df,
    group_column=event_column,
    output_path=FIGURES_DIR / "hate_v2_by_event.png",
    title="Predicción exploratoria de odio por momento electoral",
)
media_plot_df = model_viz.save_group_profile_figure(
    hate_by_media_df,
    group_column="anchor_media_handle",
    output_path=FIGURES_DIR / "hate_v2_by_anchor_media.png",
    title="Predicción exploratoria de odio por medio ancla",
    min_rows=HATE_MEDIA_MIN_ROWS,
    max_groups=20,
)
model_viz.save_overlap_figure(
    hate_overlap_df,
    FIGURES_DIR / "hostility_hate_v2_overlap.png",
)

print("Porcentaje balanceado:", f"{hate_overall_df.iloc[0]['predicted_positive_pct']:.2f}%")
print(
    "Porcentaje alta precisión:",
    f"{hate_overall_df.iloc[1]['predicted_positive_pct']:.2f}%",
)
display(hate_overall_df)
display(hate_by_event_df)
display(
    hate_by_media_df[
        hate_by_media_df["n_rows"].ge(HATE_MEDIA_MIN_ROWS)
    ].sort_values(["profile", "predicted_positive_pct"], ascending=[True, False])
)
display(hate_overlap_df)


In [ ]:
known_fn_recheck_df = refinement.recheck_known_false_negatives(
    consensus_profiles_df,
    known_fn_df,
)
known_fn_recheck_path = REPORTS_DIR / "hate_v2_known_false_negatives_recheck.csv"
known_fn_recheck_df.to_csv(known_fn_recheck_path, index=False)
print("Casos conocidos:", len(known_fn_recheck_df))
print("Recuperados por perfil balanceado:", int(known_fn_recheck_df["pred_balanced"].sum()))
print("Recuperados por perfil alta precisión:", int(known_fn_recheck_df["pred_high_precision"].sum()))
print("[OK]", known_fn_recheck_path)
display(known_fn_recheck_df.drop(columns=["tweet_id"]).head(20))


## Advertencia metodológica

1. La muestra combinada está enriquecida por lexicón y aprendizaje activo.
2. Los umbrales se seleccionan sobre scores fuera de muestra, pero dentro del mismo conjunto anotado; siguen siendo exploratorios.
3. El perfil `balanced` busca un mejor equilibrio entre precisión y recall.
4. El perfil `high_precision` sacrifica recall para reducir falsos positivos.
5. Ningún porcentaje aplicado al corpus completo debe presentarse como prevalencia real de hostilidad u odio.
6. Una evaluación formal posterior necesita un conjunto aleatorio independiente que no participe en selección, ajuste de umbral ni entrenamiento.

- El baseline de lexicón mide coincidencias de términos y no debe interpretarse como detector automático de odio.
